In [1]:
import pandas as pd
import glob
import os
import ee
import time

In [2]:
RAW_PATH = "../data/raw"
DEMOGRAPHIC_PATH = "data/demographic"
OUTPUT_PATH = "data/processed/dengue_full_final.csv"

PROJECT_ID = "dengue-ml-project"

YEARS = [2016, 2017, 2018, 2019, 2023, 2024]
MONTHS = list(range(1, 13))

In [3]:
def load_and_unify_data(path_folder):
    files = glob.glob(os.path.join(path_folder, "*.xlsx"))
    df_list = []

    for file in files:
        print(f"Cargando: {file}")
        df = pd.read_excel(file)
        df.columns = df.columns.str.strip().str.lower()
        df_list.append(df)

    df = pd.concat(df_list, ignore_index=True)
    return df

In [4]:
df = load_and_unify_data(RAW_PATH)

Cargando: ../data/raw/2017.xlsx
Cargando: ../data/raw/2016.xlsx
Cargando: ../data/raw/2024.xlsx
Cargando: ../data/raw/2019.xlsx
Cargando: ../data/raw/2023.xlsx
Cargando: ../data/raw/2018.xlsx


In [5]:
len(df)

728496

In [7]:
df = df.astype(str)

In [8]:
import pyarrow as pa

# Engañamos a Pandas registrando el tipo que él quiere borrar
try:
    class FakeExtensionType(pa.PyExtensionType):
        def __init__(self):
            pa.PyExtensionType.__init__(self, pa.null())
        def __reduce__(self):
            return FakeExtensionType, ()

    pa.register_extension_type(FakeExtensionType())
except Exception:
    pass

import pandas as pd
# Ahora corre tu código
df.to_parquet('data.parquet', compression='gzip')